In [46]:
#References: https://medium.com/@ayush.rane/experimenting-with-ml-trying-out-different-algorithms-for-one-simple-task-2431369c0223

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn import linear_model
from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.svm import SVC

In [47]:
# Load dataset
df = pd.read_csv("../data/merged_final_transformed.csv")
print(f'Shape: {df.shape}')
df.head()


Shape: (6646, 41)


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,median_age,pct_bachelors_plus,pct_graduate_degree,pct_less_than_hs,pct_white,pct_black,pct_asian,pct_hispanic,median_household_income,climate_type_short
0,2013,AK,ANCHORAGE MUNICIPALITY,2020,28.691667,NaN,NaN,NaN,NaN,NaN,...,32.8,11.7,92.5,23.2,0.1,7.9,7.8,92.0,77454,ET
1,2013,AL,JEFFERSON COUNTY,1073,41.893750,NaN,NaN,NaN,NaN,NaN,...,37.2,11.6,87.4,26.6,0.1,0.8,0.8,96.2,45429,Cfa
2,2013,AL,MADISON COUNTY,1089,37.654348,NaN,NaN,NaN,NaN,NaN,...,37.4,14.5,90.0,21.4,0.1,2.3,2.2,95.4,58434,Cfa
3,2013,AL,MOBILE COUNTY,1097,44.833333,NaN,NaN,NaN,NaN,NaN,...,36.7,7.0,83.9,33.0,0.0,1.4,1.4,97.5,43028,Cfa
4,2013,AL,MONTGOMERY COUNTY,1101,39.792727,NaN,NaN,NaN,NaN,NaN,...,34.8,12.0,85.6,26.2,0.1,1.3,1.1,96.5,44790,Cfa


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6646 entries, 0 to 6645
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     6646 non-null   int64  
 1   StateAbbr                6646 non-null   str    
 2   County name              6646 non-null   str    
 3   CountyFIPS               6646 non-null   int64  
 4   BPHIGH                   6031 non-null   float64
 5   CASTHMA                  6322 non-null   float64
 6   COPD                     6322 non-null   float64
 7   MHLTH                    6322 non-null   float64
 8   PHLTH                    6322 non-null   float64
 9   SLEEP                    3195 non-null   float64
 10  STROKE                   6322 non-null   float64
 11  STATION                  6601 non-null   str    
 12  STATION_NAME             6601 non-null   str    
 13  LATITUDE                 6601 non-null   float64
 14  LONGITUDE                6601 non-n

In [49]:
missing = df.isnull().sum() / len(df) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print('Features with missing values (% missing):')
print(missing.round(1).to_string())

Features with missing values (% missing):
SLEEP           51.9
BPHIGH           9.3
PRCP             8.5
HTDD             8.0
CLDD             7.8
TAVG             7.7
TMIN             7.4
DT32             7.4
EMNT             7.4
TMAX             7.3
EMXT             7.3
DX90             7.3
DX70             7.3
DX32             7.3
STROKE           4.9
CASTHMA          4.9
MHLTH            4.9
COPD             4.9
PHLTH            4.9
ELEVATION        0.7
LONGITUDE        0.7
LATITUDE         0.7
STATION_NAME     0.7
STATION          0.7


# Preparing the data with one hat encoding categorical 

In [50]:
# Separate feature types — keep as lists for ColumnTransformer
target = "CASTHMA"

df = df.drop(columns=["County name", "CountyFIPS", 'BPHIGH', 'SLEEP', 'COPD', 'MHLTH', 'PHLTH','STROKE'])
#Should we drop STATION? STATION_NAME?
categorical_features = df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
numerical_features = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
                      if c != target]

The datasets for targets != SLEEP are larger due to dropping sleep columns that have missing values. 

In [51]:
missing = df.isnull().sum() / len(df) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print('Features with missing values (% missing):')
print(missing.round(1).to_string())

Features with missing values (% missing):
PRCP            8.5
HTDD            8.0
CLDD            7.8
TAVG            7.7
TMIN            7.4
DT32            7.4
EMNT            7.4
DX90            7.3
TMAX            7.3
EMXT            7.3
DX70            7.3
DX32            7.3
CASTHMA         4.9
STATION         0.7
ELEVATION       0.7
LONGITUDE       0.7
LATITUDE        0.7
STATION_NAME    0.7


In [52]:
df = df.dropna(subset=["CASTHMA"])

# Asthma

CSATHMA

In [53]:

X = df.drop(target, axis = 1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')


Train: 5057 samples
Test:  1265 samples


I went through the Linear Models using sklearn: https://sklearn.org/stable/modules/linear_model.html. This is just an initial pass. There was a reason to skip classification models as said later due to having to make a second column like are we going to say "yes, you have mental health issues" or "no you dont". Then we could create a new column that we change based on values. But if that is not the case and we want to predict how bad the mental health is if you live in that state, might be more useful than a direct binary. 

In [54]:
# Creation of a dictionary to print at the end which we will sort by value 

# We use the R2 score to show the accuracy. We cannot use accuracy_score because we are
# are not using classification algorithms. The closer to 1 the R2 score is the better the
# algorithm is: https://www.geeksforgeeks.org/maths/r-squared/

R2_scores = {}
MSE_scores = {}

In [56]:
def make_preprocessor():
    """Returns a fresh ColumnTransformer for the Ames dataset."""
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
             categorical_features),
            ('num', SimpleImputer(strategy='mean'),
             numerical_features)
        ]
    )
    # Note: no remainder='passthrough' — all columns are explicitly listed

In [57]:
def create_ridge_pipeline(alpha):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        Ridge(alpha=alpha)
    )
#lab 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = np.logspace(-4, 4, 20)

cv_means = []
cv_ses   = []

for alpha in alphas:
    scores = cross_val_score(
        create_ridge_pipeline(alpha),
        X_train,
        y_train,
        cv=kf,
        scoring=make_scorer(mean_squared_error)
    )
    cv_means.append(scores.mean())
    cv_ses.append(scores.std() / np.sqrt(len(scores)))

cv_means = np.array(cv_means)
cv_ses   = np.array(cv_ses)
best_alpha_ridge = alphas[np.argmin(cv_means)]

final_ridge = create_ridge_pipeline(best_alpha_ridge)
final_ridge.fit(X_train, y_train)
test_mse_ridge = mean_squared_error(y_test, final_ridge.predict(X_test))
print(f'Ridge Test MSE: {test_mse_ridge:.4f}')
MSE_scores["Ridge"] = test_mse_ridge

y_pred = pipe.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Ridge: {accuracy}")
R2_scores["Ridge"] = accuracy

Ridge Test MSE: 0.5022
R^2 score Ridge: 0.6695785244679104


In [58]:
def create_krr_pipeline(alpha, gamma):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        KernelRidge(alpha=alpha, kernel='rbf', gamma=gamma)
    )


In [59]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
alphas_krr = np.logspace(-8, -1, num=10)
gammas_krr = np.logspace(-8, -1, num=10)

mean_mses_2d = np.zeros((len(alphas_krr), len(gammas_krr)))

for i, alpha in enumerate(alphas_krr):
    for j, gamma in enumerate(gammas_krr):
        model = create_krr_pipeline(alpha, gamma)
        scores = cross_val_score(
            model, X_train, y_train,
            cv=kf, scoring=make_scorer(mean_squared_error)
        )
        mean_mse = scores.mean()
        mean_mses_2d[i, j] = mean_mse

best_i, best_j = np.unravel_index(np.argmin(mean_mses_2d), mean_mses_2d.shape)
best_alpha_krr = alphas_krr[best_i]
best_gamma_krr = gammas_krr[best_j]

In [60]:
print(f'Best alpha: {best_alpha_krr}')
print(f'Best gamma: {best_gamma_krr}')
print(f'CV MSE:     {mean_mses_2d[best_i, best_j]:.4f}')

Best alpha: 1e-08
Best gamma: 3.5938136638046254e-07
CV MSE:     0.4682


In [72]:
# Using best gamma and alpha
pipe_krr = create_krr_pipeline(alpha=best_alpha_krr, gamma=best_gamma_krr)
pipe_krr.fit(X_train, y_train)
MSE = mean_squared_error(y_test, pipe_krr.predict(X_test))
print(f'KRR Test MSE: {MSE:.4f}')
MSE_scores["KRR"] = MSE

y_pred = pipe_krr.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score KRR: {accuracy}")
R2_scores["KRR"] = accuracy

KRR Test MSE: 0.4898
R^2 score KRR: 0.6760891309836348


In [73]:
def create_linear_regression_pipeline():
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        LinearRegression()
    )

# Check it worked
pipe_linear_r = create_linear_regression_pipeline()
pipe_linear_r.fit(X_train, y_train)
MSE = mean_squared_error(y_test, pipe_linear_r.predict(X_test))
print(f'Linear Regression test MSE: {MSE:.4f}')
MSE_scores["Linear Regression"] = MSE

y_pred = pipe_linear_r.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Linear Regression: {accuracy}")
R2_scores["Linear Regression"] = accuracy

Linear Regression test MSE: 0.4995
R^2 score Linear Regression: 0.6696281750562842


In [63]:
def create_lasso_pipeline(alpha):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        linear_model.Lasso(alpha)
    )

kf = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = np.logspace(-4, 4, 10)

cv_means = []
cv_ses   = []

for alpha in alphas:
    scores = cross_val_score(
        create_lasso_pipeline(alpha),
        X_train,
        y_train,
        cv=kf,
        scoring=make_scorer(mean_squared_error)
    )
    cv_means.append(scores.mean())
    cv_ses.append(scores.std() / np.sqrt(len(scores)))

cv_means = np.array(cv_means)
cv_ses   = np.array(cv_ses)
best_alpha_lasso = alphas[np.argmin(cv_means)]

final_lasso = create_lasso_pipeline(best_alpha_lasso)
final_lasso.fit(X_train, y_train)
test_mse_lasso = mean_squared_error(y_test, final_lasso.predict(X_test))
print(f'Lasso Test MSE: {test_mse_lasso:.4f}')
MSE_scores["Lasso"] = test_mse_lasso

y_pred = final_lasso.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Lasso: {accuracy}")
R2_scores["Lasso"] = accuracy

/Users/aj/hertie/dsa/project/project-krasss/ml_scripts/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.887e+01, tolerance: 6.680e-01
  model = cd_fast.enet_coordinate_descent(
/Users/aj/hertie/dsa/project/project-krasss/ml_scripts/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.830e+01, tolerance: 6.843e-01
  model = cd_fast.enet_coordinate_descent(
/Users/aj/hertie/dsa/project/project-krasss/ml_scripts/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want 

Lasso Test MSE: 0.5027
R^2 score Lasso: 0.6675636692127902


In [ ]:
def create_Bayesian_Ridge_pipeline():
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        linear_model.BayesianRidge()
    )

pipe_Bayesian_ridge = create_Bayesian_Ridge_pipeline()
pipe_Bayesian_ridge.fit(X_train, y_train)
MSE = mean_squared_error(y_test, pipe_Bayesian_ridge.predict(X_test))
print(f'Baysian Ridge test MSE:{MSE:.4f}')
MSE_scores["Bayesian Ridge"] = MSE

y_pred = pipe_Bayesian_ridge.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Baysian Ridge: {accuracy}")
R2_scores["Bayesian Ridge"] = accuracy

Baysian Ridge training MSE:0.5098
R^2 score Baysian Ridge: 0.6628432743697629


In [74]:
#Constant that multiplies the penalty term. Defaults to 1.0. alpha = 0 is equivalent to an ordinary least square, 
# solved by LinearRegression. For numerical reasons, using alpha = 0 
# with the LassoLars object is not advised and you should prefer the LinearRegression object.

def create_LassoLARS_pipeline(alpha):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        linear_model.LassoLars(alpha)
    )

# Check it worked
pipe_Lasso_Lars = create_LassoLARS_pipeline(alpha=1)
pipe_Lasso_Lars.fit(X_train, y_train)
MSE = mean_squared_error(y_test, pipe_Lasso_Lars.predict(X_test))
print(f'Lasso Lars test MSE: {MSE:.4f}')
MSE_scores["Lasso LARS with alpha 1"] = MSE

y_pred = pipe_Lasso_Lars.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Lasso LARS: {accuracy}")
R2_scores["Lasso LARS with alpha 1"] = accuracy

Lasso Lars test MSE: 1.5148
R^2 score Lasso LARS: -0.0018194925363257397


In [75]:
from sklearn.neighbors import KNeighborsRegressor
def create_KNN_pipeline(n_neighbors):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        KNeighborsRegressor()
    )

# Check it worked
pipe_KNN = create_KNN_pipeline(n_neighbors=5)
pipe_KNN.fit(X_train, y_train)
MSE = mean_squared_error(y_test, pipe_KNN.predict(X_test))
print(f'KNN training MSE: {MSE:.4f}')
MSE_scores["KNN with 5 neighbors"] = MSE

y_pred = pipe_KNN.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score KNN: {accuracy}")
R2_scores["KNN with 5 neighbor"] = accuracy

KNN training MSE: 0.6869
R^2 score KNN: 0.5456953857059303


In [76]:
R2_scores

{'Ridge': 0.6695785244679104,
 'KRR': 0.6760891309836348,
 'Linear Regression': 0.6696281750562842,
 'Lasso': 0.6675636692127902,
 'Bayesian Ridge': 0.6628432743697629,
 'Lasso LARS with alpha 1': -0.0018194925363257397,
 'KNN with 5 neighbor': 0.5456953857059303}

In [77]:
sorted_r2 = {key: value for key, 
               value in reversed(sorted(R2_scores.items(), 
                               key=lambda item: item[1]))}

#We sort in reverse order because the best one is the one closest to 1
sorted_r2

{'KRR': 0.6760891309836348,
 'Linear Regression': 0.6696281750562842,
 'Ridge': 0.6695785244679104,
 'Lasso': 0.6675636692127902,
 'Bayesian Ridge': 0.6628432743697629,
 'KNN with 5 neighbor': 0.5456953857059303,
 'Lasso LARS with alpha 1': -0.0018194925363257397}

In [78]:
MSE_scores

{'Ridge': 0.5022443109205582,
 'KRR': 0.4897683939540393,
 'Linear Regression': 0.49953766170833136,
 'Lasso': 0.5026592911687671,
 'Bayesian Ridge': 0.5097967491000772,
 'Lasso LARS with alpha 1': 1.5147979608753939,
 'KNN with 5 neighbors': 0.686929839632748}

In [79]:
sorted_MSE = {key: value for key, 
               value in sorted(MSE_scores.items(), 
                               key=lambda item: item[1])}

# want the one closest to 0 MSE 
sorted_MSE

{'KRR': 0.4897683939540393,
 'Linear Regression': 0.49953766170833136,
 'Ridge': 0.5022443109205582,
 'Lasso': 0.5026592911687671,
 'Bayesian Ridge': 0.5097967491000772,
 'KNN with 5 neighbors': 0.686929839632748,
 'Lasso LARS with alpha 1': 1.5147979608753939}

For CASTHMA: KRR